<a href="https://colab.research.google.com/github/ibelalov/rlmw/blob/main/rlmw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# rlmw

Single-notebook development environment for low-weight span-vector search.

Core components:

1. Binary linear algebra over F_2
2. Planted span instance generator
3. Bitset/popcount utilities
4. Direction bank construction
5. Exact discrete-gradient descent
6. Local-minimum support-intersection constraints
7. Failed-local-minimum archive
8. Neural direction model
9. Cross-entropy sampler
10. Strategy-level Q-controller
11. Exact local-minimum intersection solver
12. Training/evaluation dashboard

In [2]:
# @title
# --- 00. Environment setup ---

import os
import sys
import math
import time
import json
import random
import pathlib
import itertools
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("PyTorch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
PyTorch available: True
PyTorch: 2.10.0+cpu
CUDA available: False


In [7]:
# @title Environment detection and storage backend selection
import os


def _env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


# Small self-tests for environment-flag parsing
assert _env_flag("__RLMW_TEST_UNSET__", "0") is False
assert _env_flag("__RLMW_TEST_UNSET__", "1") is True

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

HEADLESS = _env_flag("RLMW_HEADLESS", "0")
SMOKE = _env_flag("RLMW_SMOKE", "0")
USE_DRIVE = IN_COLAB and (not HEADLESS)

if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [9]:
# @title Project paths
import os
from pathlib import Path

if USE_DRIVE:
    PROJECT_ROOT = Path("/content/drive/MyDrive") / "rlmw"
else:
    PROJECT_ROOT = Path(os.environ.get("RLMW_PROJECT_ROOT", "/tmp/rlmw"))

DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
EXPORT_DIR = PROJECT_ROOT / "exports"
CACHE_DIR = PROJECT_ROOT / "cache"

for p in [PROJECT_ROOT, DATA_DIR, RUNS_DIR, CHECKPOINT_DIR, EXPORT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("HEADLESS:", HEADLESS)
print("SMOKE:", SMOKE)
print("USE_DRIVE:", USE_DRIVE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("CACHE_DIR:", CACHE_DIR)

Project root: /content/drive/MyDrive/rlmw
Data dir: /content/drive/MyDrive/rlmw/data
Runs dir: /content/drive/MyDrive/rlmw/runs
Checkpoint dir: /content/drive/MyDrive/rlmw/checkpoints
Export dir: /content/drive/MyDrive/rlmw/exports
Cache dir: /content/drive/MyDrive/rlmw/cache


## 01. Binary linear algebra over F_2

In [ ]:
# @title GF(2) utilities (exact, uint8/binary)
import numpy as np


def as_binary_uint8(x, *, name="array", copy=True) -> np.ndarray:
    """Return `x` as a uint8 NumPy array containing only 0/1."""
    arr = np.array(x, copy=True) if copy else np.asarray(x)

    if arr.dtype == np.bool_:
        return arr.astype(np.uint8, copy=copy)

    if not np.issubdtype(arr.dtype, np.integer):
        raise TypeError(f"{name} must have boolean or integer dtype, got {arr.dtype!r}")

    mask = (arr == 0) | (arr == 1)
    if arr.size and not mask.all():
        bad = np.unique(arr[~mask])
        raise ValueError(f"{name} must be binary (0/1) only; found values {bad.tolist()}")

    return arr.astype(np.uint8, copy=copy)


def hamming_weight(x) -> int:
    arr = as_binary_uint8(x, name="x", copy=False)
    return int(arr.sum(dtype=np.int64))


def gf2_matvec(A, u) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    u_bin = as_binary_uint8(u, name="u", copy=False)

    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")
    if u_bin.ndim != 1:
        raise ValueError(f"u must be 1D, got shape {u_bin.shape}")
    if A_bin.shape[1] != u_bin.shape[0]:
        raise ValueError(f"shape mismatch: A is {A_bin.shape}, u is {u_bin.shape}")

    y = (A_bin.astype(np.int64) @ u_bin.astype(np.int64)) & 1
    return y.astype(np.uint8)


def gf2_matmul(A, B) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=False)
    B_bin = as_binary_uint8(B, name="B", copy=False)

    if A_bin.ndim != 2 or B_bin.ndim != 2:
        raise ValueError(f"A and B must be 2D, got {A_bin.shape} and {B_bin.shape}")
    if A_bin.shape[1] != B_bin.shape[0]:
        raise ValueError(f"shape mismatch: A is {A_bin.shape}, B is {B_bin.shape}")

    C = (A_bin.astype(np.int64) @ B_bin.astype(np.int64)) & 1
    return C.astype(np.uint8)


def gf2_rank(A) -> int:
    M = as_binary_uint8(A, name="A", copy=True)
    if M.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {M.shape}")

    rows, cols = M.shape
    r = 0
    for c in range(cols):
        pivot = None
        for i in range(r, rows):
            if M[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            continue

        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]

        for i in range(r + 1, rows):
            if M[i, c] == 1:
                M[i, :] ^= M[r, :]

        r += 1
        if r == rows:
            break

    return int(r)


def gf2_random_matrix(rows, cols, rng=None, density=0.5) -> np.ndarray:
    if rows < 0 or cols < 0:
        raise ValueError("rows and cols must be nonnegative")
    if not (0.0 <= density <= 1.0):
        raise ValueError("density must be in [0, 1]")

    rng = np.random.default_rng() if rng is None else rng
    A = (rng.random((rows, cols)) < density).astype(np.uint8)
    return as_binary_uint8(A, name="A", copy=False)


def gf2_random_invertible(n, rng=None, max_tries=1000) -> np.ndarray:
    if n < 0:
        raise ValueError("n must be nonnegative")
    if max_tries <= 0:
        raise ValueError("max_tries must be positive")

    rng = np.random.default_rng() if rng is None else rng

    for _ in range(max_tries):
        Q = gf2_random_matrix(n, n, rng=rng, density=0.5)
        if gf2_rank(Q) == n:
            return Q

    raise RuntimeError(f"failed to sample invertible {n}x{n} matrix within {max_tries} tries")


def gf2_inverse(A) -> np.ndarray:
    A_bin = as_binary_uint8(A, name="A", copy=True)
    if A_bin.ndim != 2:
        raise ValueError(f"A must be 2D, got shape {A_bin.shape}")

    n, m = A_bin.shape
    if n != m:
        raise ValueError(f"A must be square, got shape {A_bin.shape}")

    I = np.eye(n, dtype=np.uint8)
    aug = np.concatenate([A_bin, I], axis=1)

    for c in range(n):
        pivot = None
        for i in range(c, n):
            if aug[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            raise ValueError("A is singular over F_2")

        if pivot != c:
            aug[[c, pivot]] = aug[[pivot, c]]

        for i in range(n):
            if i != c and aug[i, c] == 1:
                aug[i, :] ^= aug[c, :]

    inv = aug[:, n:]
    return as_binary_uint8(inv, name="A_inv", copy=False)


def support_intersection(c, d) -> int:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    d_bin = as_binary_uint8(d, name="d", copy=False)

    if c_bin.ndim != 1 or d_bin.ndim != 1:
        raise ValueError(f"c and d must be 1D, got {c_bin.shape} and {d_bin.shape}")
    if c_bin.shape != d_bin.shape:
        raise ValueError(f"shape mismatch: c is {c_bin.shape}, d is {d_bin.shape}")

    return int(np.bitwise_and(c_bin, d_bin).sum(dtype=np.int64))


def directional_delta(c, d) -> int:
    c_bin = as_binary_uint8(c, name="c", copy=False)
    d_bin = as_binary_uint8(d, name="d", copy=False)

    if c_bin.ndim != 1 or d_bin.ndim != 1:
        raise ValueError(f"c and d must be 1D, got {c_bin.shape} and {d_bin.shape}")
    if c_bin.shape != d_bin.shape:
        raise ValueError(f"shape mismatch: c is {c_bin.shape}, d is {d_bin.shape}")

    return hamming_weight(d_bin) - 2 * support_intersection(c_bin, d_bin)


def run_f2_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    # Binary/uint8 conversion checks
    v = as_binary_uint8([0, 1, 1, 0], name="v")
    assert v.dtype == np.uint8 and np.isin(v, [0, 1]).all()
    b = as_binary_uint8(np.array([True, False, True]), name="b")
    assert b.dtype == np.uint8 and np.array_equal(b, np.array([1, 0, 1], dtype=np.uint8))

    try:
        as_binary_uint8(np.array([0, 2], dtype=np.int64), name="bad")
        raise AssertionError("expected ValueError for non-binary integers")
    except ValueError:
        pass

    try:
        as_binary_uint8(np.array([0.0, 1.0]), name="float_bad")
        raise AssertionError("expected TypeError for non-integer dtype")
    except TypeError:
        pass


    for overflow_bad in [
        np.array([256], dtype=np.int64),
        np.array([-256], dtype=np.int64),
        np.array([0, 1, 257], dtype=np.int64),
        np.array([256], dtype=np.uint64),
    ]:
        try:
            as_binary_uint8(overflow_bad, name="overflow_bad")
            raise AssertionError("expected ValueError for overflow-prone non-binary integers")
        except ValueError:
            pass

    # Edge vectors
    z = as_binary_uint8(np.zeros(6, dtype=np.uint8), name="z")
    oh = as_binary_uint8(np.array([0, 0, 1, 0, 0, 0], dtype=np.uint8), name="oh")
    ones = as_binary_uint8(np.ones(6, dtype=np.uint8), name="ones")
    assert hamming_weight([0, 1, 1, 0]) == 2
    y_list = gf2_matvec([[1, 0], [1, 1]], [1, 1])
    assert y_list.dtype == np.uint8 and np.array_equal(y_list, np.array([1, 0], dtype=np.uint8))

    assert hamming_weight(z) == 0
    assert hamming_weight(oh) == 1
    assert hamming_weight(ones) == 6

    # Matvec: compare with direct parity loops
    for _ in range(20):
        n = int(rng.integers(1, 7))
        r = int(rng.integers(1, 7))
        A = gf2_random_matrix(n, r, rng=rng, density=0.5)
        u = gf2_random_matrix(r, 1, rng=rng, density=0.5).reshape(-1)
        y = gf2_matvec(A, u)

        y_ref = np.zeros(n, dtype=np.uint8)
        for i in range(n):
            s = 0
            for j in range(r):
                s ^= int(A[i, j] & u[j])
            y_ref[i] = s

        assert y.dtype == np.uint8 and y.shape == (n,)
        assert np.array_equal(y, y_ref)

    # Matmul: compare with direct parity loops
    for _ in range(20):
        n = int(rng.integers(1, 6))
        r = int(rng.integers(1, 6))
        k = int(rng.integers(1, 6))
        A = gf2_random_matrix(n, r, rng=rng, density=0.5)
        B = gf2_random_matrix(r, k, rng=rng, density=0.5)
        C = gf2_matmul(A, B)

        C_ref = np.zeros((n, k), dtype=np.uint8)
        for i in range(n):
            for j in range(k):
                s = 0
                for t in range(r):
                    s ^= int(A[i, t] & B[t, j])
                C_ref[i, j] = s

        assert C.dtype == np.uint8 and C.shape == (n, k)
        assert np.array_equal(C, C_ref)

    # Rank edge cases
    Z = np.zeros((4, 5), dtype=np.uint8)
    assert gf2_rank(Z) == 0

    I5 = np.eye(5, dtype=np.uint8)
    assert gf2_rank(I5) == 5

    D = np.array([
        [1, 0, 1, 1],
        [1, 0, 1, 1],
        [0, 1, 1, 0],
    ], dtype=np.uint8)
    assert gf2_rank(D) == 2

    for _ in range(20):
        n = int(rng.integers(1, 7))
        m = int(rng.integers(1, 7))
        A = gf2_random_matrix(n, m, rng=rng, density=0.5)
        rnk = gf2_rank(A)
        assert isinstance(rnk, int)
        assert 0 <= rnk <= min(n, m)

    # Small rectangular matrices
    A_rect = gf2_random_matrix(2, 5, rng=rng)
    B_rect = gf2_random_matrix(5, 3, rng=rng)
    C_rect = gf2_matmul(A_rect, B_rect)
    assert C_rect.shape == (2, 3)
    assert C_rect.dtype == np.uint8 and np.isin(C_rect, [0, 1]).all()

    # Invertible sampling and inverse checks
    for n in [1, 2, 3, 5]:
        Q = gf2_random_invertible(n, rng=rng, max_tries=2000)
        Q_inv = gf2_inverse(Q)
        I = np.eye(n, dtype=np.uint8)
        assert np.array_equal(gf2_matmul(Q, Q_inv), I)
        assert np.array_equal(gf2_matmul(Q_inv, Q), I)

    # singular / non-square inverse errors
    try:
        gf2_inverse(np.array([[1, 0, 1]], dtype=np.uint8))
        raise AssertionError("expected ValueError for non-square matrix")
    except ValueError:
        pass

    try:
        gf2_inverse(np.array([[1, 1], [1, 1]], dtype=np.uint8))
        raise AssertionError("expected ValueError for singular matrix")
    except ValueError:
        pass

    # Directional delta identities
    for _ in range(40):
        n = int(rng.integers(1, 20))
        c = gf2_random_matrix(n, 1, rng=rng).reshape(-1)
        d = gf2_random_matrix(n, 1, rng=rng).reshape(-1)

        delta = directional_delta(c, d)
        lhs1 = hamming_weight(c ^ d) - hamming_weight(c)
        lhs2 = hamming_weight(d) - 2 * support_intersection(c, d)

        assert delta == lhs1
        assert delta == lhs2

    # explicit edge cases for directional_delta
    assert directional_delta(z, z) == 0
    assert directional_delta(z, oh) == 1
    assert directional_delta(ones, ones) == -6

    print("run_f2_self_tests: all tests passed")

In [ ]:
run_f2_self_tests()

## 02. Planted span-instance generator

In [ ]:
# @title Planted span-instance generator (exact, uint8/binary)
from dataclasses import dataclass


@dataclass
class SpanInstance:
    A: np.ndarray
    c_star: np.ndarray
    u_star: np.ndarray
    W: int
    metadata: dict

    def matvec(self, u) -> np.ndarray:
        return gf2_matvec(self.A, u)

    def weight(self, c) -> int:
        return hamming_weight(c)

    def verify(self) -> None:
        A = as_binary_uint8(self.A, name="A", copy=False)
        c_star = as_binary_uint8(self.c_star, name="c_star", copy=False)
        u_star = as_binary_uint8(self.u_star, name="u_star", copy=False)

        if A.ndim != 2:
            raise ValueError(f"A must be 2D, got shape {A.shape}")
        N, r = A.shape

        if c_star.ndim != 1 or c_star.shape != (N,):
            raise ValueError(f"c_star must have shape ({N},), got {c_star.shape}")
        if u_star.ndim != 1 or u_star.shape != (r,):
            raise ValueError(f"u_star must have shape ({r},), got {u_star.shape}")

        if hamming_weight(u_star) == 0:
            raise ValueError("u_star must be nonzero")
        if int(self.W) != hamming_weight(c_star):
            raise ValueError("W must equal hamming_weight(c_star)")

        if not np.array_equal(gf2_matvec(A, u_star), c_star):
            raise ValueError("constraint violated: A u_star != c_star")


def generate_planted_span_instance(
    N: int,
    r: int,
    W: int,
    seed=None,
    density: float = 0.5,
    hide: bool = True,
) -> SpanInstance:
    if N <= 0:
        raise ValueError("N must be positive")
    if r <= 0:
        raise ValueError("r must be positive")
    if not (1 <= W <= N):
        raise ValueError("W must satisfy 1 <= W <= N")
    if not (0.0 <= density <= 1.0):
        raise ValueError("density must be in [0, 1]")

    rng = np.random.default_rng(seed)

    support = rng.choice(N, size=W, replace=False)
    c_star = np.zeros(N, dtype=np.uint8)
    c_star[support] = 1

    u_star = np.zeros(r, dtype=np.uint8)
    while hamming_weight(u_star) == 0:
        u_star = (rng.random(r) < 0.5).astype(np.uint8)

    one_positions = np.flatnonzero(u_star)
    k = int(rng.choice(one_positions))

    A = gf2_random_matrix(N, r, rng=rng, density=density)

    parity = np.zeros(N, dtype=np.uint8)
    for j in one_positions:
        if j != k:
            parity ^= A[:, j]
    A[:, k] = c_star ^ parity

    if hide:
        Q = gf2_random_invertible(r, rng=rng)
        Q_inv = gf2_inverse(Q)
        A = gf2_matmul(A, Q)
        u_star = gf2_matvec(Q_inv, u_star)

    if not np.array_equal(gf2_matvec(A, u_star), c_star):
        raise RuntimeError("internal construction error: A u_star != c_star")

    metadata = {
        "N": int(N),
        "r": int(r),
        "W": int(W),
        "seed": seed,
        "density": float(density),
        "hide": bool(hide),
    }
    inst = SpanInstance(A=A, c_star=c_star, u_star=u_star, W=int(W), metadata=metadata)
    inst.verify()
    return inst


def run_planted_instance_self_tests(seed=0) -> None:
    rng = np.random.default_rng(seed)

    for hide in [False, True]:
        for _ in range(8):
            N = int(rng.integers(4, 13))
            r = int(rng.integers(2, 11))
            W = int(rng.integers(1, N + 1))
            density = float(rng.uniform(0.0, 1.0))
            s = int(rng.integers(0, 10**9))

            inst = generate_planted_span_instance(N=N, r=r, W=W, seed=s, density=density, hide=hide)
            inst.verify()

            assert inst.weight(inst.c_star) == W
            assert np.array_equal(inst.matvec(inst.u_star), inst.c_star)

            for arr in [inst.A, inst.c_star, inst.u_star]:
                assert arr.dtype == np.uint8
                assert np.isin(arr, [0, 1]).all()

    params = dict(N=10, r=6, W=3, seed=12345, density=0.4, hide=True)
    a = generate_planted_span_instance(**params)
    b = generate_planted_span_instance(**params)
    assert np.array_equal(a.A, b.A)
    assert np.array_equal(a.c_star, b.c_star)
    assert np.array_equal(a.u_star, b.u_star)

    invalid_cases = [
        dict(N=10, r=5, W=0, seed=0, density=0.5, hide=False),
        dict(N=10, r=5, W=11, seed=0, density=0.5, hide=False),
        dict(N=0, r=5, W=1, seed=0, density=0.5, hide=False),
        dict(N=10, r=0, W=1, seed=0, density=0.5, hide=False),
        dict(N=10, r=5, W=1, seed=0, density=-0.1, hide=False),
        dict(N=10, r=5, W=1, seed=0, density=1.1, hide=False),
    ]
    for kwargs in invalid_cases:
        try:
            generate_planted_span_instance(**kwargs)
            raise AssertionError(f"expected ValueError for invalid kwargs={kwargs}")
        except ValueError:
            pass

    print("run_planted_instance_self_tests: all tests passed")

In [ ]:
run_planted_instance_self_tests()